# CS2 ShadowPro — demo data exploration

What's inside one parsed pro demo (`parsed_sample/*.parquet`, produced by `parse_one_demo.py`).

The goal is to answer: **which pieces of a "situation" (map area, side, economy, NvN, phase, utility, time remaining) can I build from what awpy already gives me, and which require asking awpy for more fields?**

Each section: what the DataFrame is → sample rows → quick visual.

## 0. Setup

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

pl.Config.set_tbl_rows(15)
pl.Config.set_tbl_cols(20)

DATA = Path('parsed_sample')
frames = {p.stem: pl.read_parquet(p) for p in sorted(DATA.glob('*.parquet'))}
{name: (df.height, df.width) for name, df in frames.items()}

## 1. `rounds` — match structure

One row per round. Gives round boundaries (`start` → `end` ticks), winner, win reason, bomb plant tick/site. This is the scaffolding every other DataFrame joins onto via `round_num`.

In [ ]:
rounds = frames['rounds']
rounds

In [ ]:
counts = rounds.group_by(['winner', 'reason']).len().sort('len', descending=True)
labels = [f'{w}/{r}' for w, r in zip(counts['winner'], counts['reason'])]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, counts['len'])
ax.set_title(f'Round outcomes ({rounds.height} rounds)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

## 2. `player_round_totals` — roster

Players and how many rounds they appeared on each side. Useful to confirm the 10 match players + map how steamids relate to names.

In [ ]:
frames['player_round_totals']

## 3. `ticks` — per-frame player snapshots (the core)

Every tick × every player = one row. This is the giant table (~1.6M rows). At default awpy settings it contains:

- position: `X`, `Y`, `Z`
- `health`
- `side` (ct / t)
- `place` — **named map area** (e.g. "Apartments", "BSite") — this is your "map area" bucket for situation matching
- identity: `steamid`, `name`, `round_num`, `tick`

### ⚠️ What's missing for situation matching

The default parse does **not** include `money` / `balance`, `active_weapon`, `armor`, `has_defuser`, `flash_duration`. You'll need to pass `player_props=[...]` to `awpy.Demo(...)` when you reparse. Verify by checking the columns below.

In [ ]:
ticks = frames['ticks']
print(f'ticks: {ticks.height:,} rows × {ticks.width} cols')
print('columns:', ticks.columns)
ticks.head(5)

### 3a. Map areas — distinct `place` values

This is your ready-made spatial bucketing for Mirage. Compare this list against what you actually care about (apps, palace, mid, B short, etc.).

In [ ]:
places = ticks.group_by('place').len().sort('len', descending=True)
places

In [ ]:
top = places.head(20)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top['place'], top['len'])
ax.invert_yaxis()
ax.set_title('Player-ticks by map area (top 20)')
plt.tight_layout(); plt.show()

### 3b. Derived: alive-count timeline for round 1

`ticks.health > 0` grouped by tick & side → instant NvN count. This is the raw input for "player count" as a situation feature — no extra parsing needed.

In [ ]:
r1 = ticks.filter(pl.col('round_num') == 1).filter(pl.col('health') > 0)
alive = r1.group_by(['tick', 'side']).len().sort('tick')
fig, ax = plt.subplots(figsize=(10, 4))
for side, color in [('ct', 'steelblue'), ('t', 'orange')]:
    sub = alive.filter(pl.col('side') == side).sort('tick')
    ax.plot(sub['tick'], sub['len'], label=side.upper(), color=color)
ax.set_xlabel('tick'); ax.set_ylabel('alive')
ax.set_title('Round 1 — alive count per side')
ax.legend(); plt.tight_layout(); plt.show()

## 4. `kills` — full kill events (47 cols)

Very rich: attacker + victim + assister positions, weapon, hitgroup, flags (`attackerblind`, `thrusmoke`, `noscope`, `headshot`), `distance`. Each row is one death.

In [ ]:
kills = frames['kills']
kills.head(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for side, color in [('ct', 'steelblue'), ('t', 'orange')]:
    sub = kills.filter(pl.col('attacker_side') == side)
    ax.scatter(sub['attacker_X'], sub['attacker_Y'], c=color, s=25, alpha=0.7, label=f'attacker {side.upper()}')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('Kill attacker positions (Mirage coords)')
ax.legend(); plt.show()

In [ ]:
w = kills.group_by('weapon').len().sort('len', descending=True)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(w['weapon'], w['len'])
plt.xticks(rotation=45, ha='right')
ax.set_title('Kills by weapon'); plt.tight_layout(); plt.show()

In [ ]:
print(f'headshot rate: {kills["headshot"].mean():.1%}')
print(f'avg distance:  {kills["distance"].mean():.0f} units')
print(f'through smoke: {kills["thrusmoke"].sum()}')
print(f'through flash: {kills["attackerblind"].sum()}')

## 5. `damages` — every damage tick

Finer-grained than kills: a shot that hits but doesn't kill still shows up here. Useful for utility damage (HE impact, molotov ticks) — look at `weapon`.

In [ ]:
dmg = frames['damages']
dmg.head(3)

In [ ]:
hg = dmg.group_by('hitgroup').agg(pl.col('dmg_health_real').sum()).sort('dmg_health_real', descending=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(hg['hitgroup'], hg['dmg_health_real'])
ax.set_title('Total real HP damage by hitgroup'); plt.tight_layout(); plt.show()

## 6. `shots` — every bullet fired

3k+ rows. Weapon + shooter position + tick. Useful for peek-timing analysis; probably not a first-class situation input.

In [ ]:
shots = frames['shots']
shots.head(3)

In [ ]:
sw = shots.group_by('weapon').len().sort('len', descending=True).head(15)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(sw['weapon'], sw['len'])
plt.xticks(rotation=45, ha='right')
ax.set_title('Shots by weapon (top 15)'); plt.tight_layout(); plt.show()

## 7. `footsteps` — player steps (16k rows)

One row per audible step. Gives a coarser position trail than `ticks` but still plenty for a heatmap.

In [ ]:
steps = frames['footsteps']
sample = steps.sample(n=min(3000, steps.height), seed=42)
fig, ax = plt.subplots(figsize=(8, 8))
for side, color in [('ct', 'steelblue'), ('t', 'orange')]:
    sub = sample.filter(pl.col('player_side') == side)
    ax.scatter(sub['player_X'], sub['player_Y'], c=color, s=4, alpha=0.3, label=side.upper())
ax.set_aspect('equal'); ax.set_title('Footstep positions (3k-row sample)')
ax.legend(); plt.show()

## 8. `grenades` — full flight trajectories

**1.9M rows** — every tick of every in-flight grenade. One `entity_id` = one throw. For "utility used in this situation" you probably don't need the trajectory; you need landing positions (see `smokes` / `infernos` below) and throw events.

In [ ]:
gren = frames['grenades']
gren.group_by('grenade_type').len().sort('len', descending=True)

In [ ]:
# Draw one smoke's flight trajectory. The `-Projectile` variants are in-flight;
# their counterparts (CSmokeGrenade, CMolotovGrenade, ...) are the landed entities.
s = gren.filter(pl.col('grenade_type') == 'CSmokeGrenadeProjectile')
ids = s['entity_id'].unique().to_list()
if ids:
    traj = s.filter(pl.col('entity_id') == ids[0]).sort('tick')
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(traj['X'], traj['Y'], '-o', ms=3)
    ax.set_aspect('equal')
    ax.set_title(f'Example smoke trajectory (entity {ids[0]}, {traj.height} points)')
    plt.show()

## 9. `smokes` & `infernos` — active utility zones

One row per smoke/molotov with `start_tick`, `end_tick`, and the final `X, Y, Z`. At any given tick, a smoke is "active" iff `start_tick ≤ t ≤ end_tick`. This is exactly what you'd use to answer "what utility was alive in this situation?"

In [ ]:
smokes = frames['smokes']
infernos = frames['infernos']
print(f'smokes: {smokes.height}   infernos: {infernos.height}')
smokes.head(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(smokes['X'], smokes['Y'], c='gray', s=60, alpha=0.6, label='smoke')
ax.scatter(infernos['X'], infernos['Y'], c='red', s=60, alpha=0.6, label='inferno')
ax.set_aspect('equal'); ax.set_title('Utility landing positions')
ax.legend(); plt.show()

## 10. `bomb` — plant / defuse / explode events

Small but critical: the `bomb_planted` tick is how you split round phase into pre-plant vs post-plant. `bombsite` is A or B.

In [ ]:
bomb = frames['bomb']
print(bomb.group_by('event').len())
bomb.head(10)

In [ ]:
plants = bomb.filter(pl.col('event') == 'bomb_planted')
fig, ax = plt.subplots(figsize=(6, 6))
for site in plants['bombsite'].unique().to_list():
    sub = plants.filter(pl.col('bombsite') == site)
    ax.scatter(sub['X'], sub['Y'], s=80, label=f'Site {site}')
ax.set_aspect('equal'); ax.set_title('Bomb plant positions')
ax.legend(); plt.show()

## 11. `server_cvars` — server config timeline

Server variable changes (timeouts, round time, friendly fire flags, etc.). Usually irrelevant for situation matching — included for completeness. Skim and move on.

In [ ]:
frames['server_cvars'].head(10)

## 12. Synthesis — situation-feature availability

Which pieces of your "situation" definition can we already build, vs. what needs a richer parse?

| Situation dimension | Source | Status |
|---|---|---|
| **Map area** | `ticks.place` (also `*_place` on kills/damages/footsteps) | ✅ ready |
| **Side (CT/T)** | `ticks.side` | ✅ ready |
| **Player count (NvN)** | derive: count `ticks.health > 0` per tick per side | ✅ derivable |
| **Round phase (pre/post-plant)** | compare current tick to `rounds.bomb_plant` | ✅ derivable |
| **Utility on the ground** | `smokes` + `infernos` (active if `start_tick ≤ t ≤ end_tick`) | ✅ derivable |
| **Time remaining** | derive from `rounds.freeze_end` / `rounds.end` / `rounds.bomb_plant` | ✅ derivable |
| **Economy tier** | NOT in default `ticks` | ❌ needs `player_props=['balance']` on reparse |
| **Equipment** (rifle/SMG/AWP/pistol) | NOT in default `ticks` | ❌ needs `player_props=['active_weapon', 'armor', ...]` |
| **Flashed / utility held** | NOT in default `ticks` | ❌ needs extra player_props |

### Recommended next step

1. Extend `parse_one_demo.py` to pass the extra `player_props` into `awpy.Demo(...)`.
2. Re-run on one demo, confirm the extra columns land in `ticks`.
3. *Then* design the `Situation` schema (probably: tick-sampled snapshots at freeze_end + every N seconds + on every kill / plant / defuse) and write the extractor.